## See "Hard Mining Negatives for Semantic Similarity"

https://www.kaggle.com/code/jithinanievarghese/hard-mining-negatives-for-semantic-similarity#Load-Data-and-preprocess-data

In [17]:
import os
import sys
PROJECT_ROOT = os.path.abspath(os.path.join(
 os.getcwd(),
 os.pardir+'/playground')
)
#### only add it once
if (PROJECT_ROOT not in sys.path):
 sys.path.append(PROJECT_ROOT)

import utils as ut
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
import numpy as np 
import pandas as pd 
import csv

import os
import json

from tqdm.auto import tqdm  # so we see progress bar
from datetime import datetime

import torch

from numba import njit
import numpy as np


from sentence_transformers import LoggingHandler, SentenceTransformer

class index_values():
    """
    A class that provides indexing functionality for values.

    Attributes:
        filename (str): The filename to save the indexer object.

    Methods:
        getval(index): Returns the string value corresponding to the given index.
        getindex(val): Returns the int index corresponding to the given value.
        set(strings): Sets the indexer based on the unique strings provided.
    """

    def __init__(self, filename=None):
        """
        Initializes an instance of the index_values class.

        Args:
            filename (str): The filename to save the indexer object.
        """
        self.filename = filename  # used to save this indexer object
        self.__load()

    def getval(self, index):
        """
        Returns the value corresponding to the given index.

        Args:
            index (int): The index to retrieve the value for.

        Returns:
            The value corresponding to the given index.
        """
        return self.indexer_reverse[index]
    
    def getindex(self, val):
        """
        Returns the index corresponding to the given value.

        Args:
            val: The value to retrieve the index for.

        Returns:
            The index corresponding to the given value.
        """
        return self.indexer[val]

    def set(self, strings):
        """
        Sets the indexer based on the unique strings provided.

        Args:
            strings (list): A list of strings to set the indexer with.
        """
        unique_strings = list(set(strings))
        self.indexer = {val: index for index, val in enumerate(unique_strings)}
        self.indexer_reverse = {index: val for val, index in self.indexer.items()}
        self.__store()

    def __load(self):
        """
        Loads the indexer from the specified file, if it exists.
        """
        if self.filename is None:
            return
        
        self.indexer = {}
        self.indexer_reverse = {}
        
        if os.path.exists(self.filename):
            with open(self.filename) as f:
                self.indexer = json.load(f)
                self.indexer_reverse = {index: val for val, index in self.indexer.items()}

    def __store(self):
        """
        Stores the indexer in the specified file.
        """
        if self.filename is None:
            return
        with open(self.filename, 'w') as f:
            json.dump(self.indexer, f)

def preprocess_text(text):
    """
    clean white space and lower case the text
    """
    return " ".join(text.split()).lower()


def getsentencelists(df,cols):
    '''
    param: df dataframe
    param: cols list of columns in dataframe to return lists from
    return: tuple of lists, each list is a string of all strings in column
     of InputExample objects
     ex.
     cols=['positive','anchor']
    positives, anchors = getsentencelists(df,cols)
    ''' 
    res={}  
    for col in cols:
        res[col]=[]
    for _,row in tqdm(df.iterrows()):
        for col in cols:
            res[col].append(row[col])
    return (res[col] for col in cols)


### Hard negative mine the positives for similar positives
### This assummes the model has been fine tuned on the dataset first
# Should I take absolute value of cosign similarity so it goes from 0-1?
#the following runs on GPU
def get_scores_processed(scores, high=0.65):
    """
    Process the scores and generate a mask based on a threshold.

    Parameters:
    scores (torch.Tensor): The input scores.
    high (float, optional): The threshold value. Defaults to 0.65.

    Returns:
    tuple: A tuple containing two numpy arrays - the processed scores and the mask.

    """
    
    # Get the entries below high
    scores_mask = (scores < high).to(scores.device)

    # Set diagonal to False (don't want true positive included in the hard negatives)
    mask = (torch.eye(scores_mask.shape[0], scores_mask.shape[0]) > 0).to(scores.device)
    scores_mask.masked_fill_(mask, False)
   
    return (scores.cpu().detach().numpy(), scores_mask.cpu().detach().numpy())


@njit
def get_hard_negatives_CPU(scores,scores_mask,positives,low=0.5, high=0.65, topn=5):
    """
    Train a sentencetransformer model, get its average similarity score, use range around that average for hard
    negatives
    expects scores to be nxn matrix of similarity scores, nparray
    expects positives to be a list of n strings
    expects high to be floats denoting the max acceptable similarity score
    topn: int, number of hard negatives to return from torch.top_k
    Get pairs of indices with low<= score <= high
    returns: list of list of positives whose similarity score is between low and high
    """
    
      # use scores_mask to select hard negatives
    hard_negatives=[]
    hn_found=0
    poor_hn_found=0

    # cntr=0
    for i,row in enumerate(scores_mask):
        # get all the positives and their scores
        # #make sure none of these positives are the same as the true positive
        # #this happens when you derive multiple queries from the same positive
        
        candidates=[(scores[i,j],positives[j]) for j in range(len(row)) if row[j]==True]     
 
        #sort by score      
        candidates.sort(key=lambda x: x[0],reverse=True)
 
        #find and remove all duplicates, do not include the true positive in candidates
        seen = set()   #empty
        seen.add(positives[i])  #do not want to return the true positive
        candidates= [tup for tup in candidates if not (tup[1] in seen or seen.add(tup[1]))]
        
        #we want the top n only so just consider those
        candidates=candidates[:topn]
        res = [pos for score, pos in candidates if score >= low ]

        if(len(res)==0):
            #nothing found between high and low, take the highest 1 seen and return it
            res = [pos for score, pos in candidates[:1]]
            if(len(res)==0):
                print(f'res=[] for row {i}')
            poor_hn_found+=1
        else:
            hn_found+=1
        
        hard_negatives.append(res)
        # cntr+=1
        # if(cntr%1000==0):
        #     print(f"{cntr} rows processed")
        
    print(f"hn_found={hn_found}, poor_hn_found={poor_hn_found}")
    return hard_negatives

In [18]:

# dataset = 'trn'
dataset='tst'

startTime = datetime.now()

#setup logging
import logging
logger = logging.getLogger(__name__)
logging.basicConfig(filename=f"LOG_{ut.modelname.split('/')[-1]}_HNM.log", encoding='utf-8', level=logging.DEBUG, filemode="w")
#### get data
logger.info('### loading data')

def get_and_save_hard_negatives(dataset,high=.65,low=0.5,topn=3):
    df = pd.read_json(f'../data/{dataset}.json')
    df.drop_duplicates(subset=['anchor', 'positive'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    df.anchor = df['anchor'].apply(lambda x: preprocess_text(x))
    df.positive = df['positive'].apply(lambda x: preprocess_text(x))

    #### get the columns of interest
    logger.info('### getting the columns of interest')

    cols=['positive','anchor']
    positives, anchors = getsentencelists(df,cols)

    # print(f'There are {len(df)} rows but the positive column has only {df['positive'].nunique()} unique values')

    #### generate the embeddings
    logger.info('### generating the embeddings')

    # Load pre-trained Sentence Transformer Model. It will be downloaded automatically
    logger.info(f'### loading sentencetransformer {ut.modelname}')

    #expect a trained model to be in the models directory, the training will help with hard negative mining
    model = SentenceTransformer(f"models/{ut.modelname.split('/')[-1]}",device="cuda:0" if torch.cuda.is_available() else "cpu",)

    # Use "convert_to_tensor=True" to keep the tensors on GPU (if available)
    positive_embeddings = model.encode(positives, convert_to_tensor=True)
    anchor_embeddings = model.encode(anchors, convert_to_tensor=True)

    logger.info(f'### generate similarity score matrix')

    # We use cosine-similarity 
    scores=model.similarity(anchor_embeddings, positive_embeddings)

    #to save memory do the following
    del positive_embeddings, anchor_embeddings
    ut.clean_up()

    # logger.info(f'###save for ease of debugging')
    # torch.save(scores,'scores.pt')

    # import pickle
    # with open("positives", "wb") as fp:   #Pickling
    #     pickle.dump(positives, fp)

    logger.info(f'### #convert all string values to their index to save space')
    indexer=index_values(None)
    indexer.set(positives)
    positives1=[indexer.getindex(val) for val in positives]

    #get the hard negatives
    positives1=[indexer.getindex(val) for val in positives]
    vals=get_scores_processed(scores,high)
    ghn_int=get_hard_negatives_CPU(*(vals),positives1,high=high,low=0.5,topn=3)

    # NOTE!!!!! only saving the first out of the list for each row!
    ghn_strs=[indexer.getval(val[0]) for val in ghn_int]

    df['negative']=ghn_strs
    df.reset_index(drop=True,inplace=True) #make sure the indexes are continuous

    try:
        df.drop(columns=['most_dissimilar_context'],inplace=True)
    except:
        pass

    #save to disk
    df.to_json(f'../data/{dataset}_with_hard_negatives.json',orient="records")
    return 

datasets=['trn','tst','eval']
for dataset in datasets:
    get_and_save_hard_negatives(dataset)

32621it [00:00, 35475.65it/s]
Batches: 100%|██████████| 1020/1020 [00:08<00:00, 116.89it/s]


hn_found=18113, poor_hn_found=14508


2633it [00:00, 34475.60it/s]
Batches: 100%|██████████| 83/83 [00:01<00:00, 47.58it/s] 


hn_found=1408, poor_hn_found=1225


2641it [00:00, 32656.42it/s]
Batches: 100%|██████████| 83/83 [00:01<00:00, 42.04it/s] 


hn_found=1166, poor_hn_found=1475


# Junk

In [20]:
test_dataset.column_names
test_dataset.features.type 
# test_dataset[:5]

StructType(struct<anchor: string, positive: string, negative: string>)

In [19]:
train_dataset.column_names
train_dataset.features.type 
# train_dataset[:5]

StructType(struct<anchor: string, positive: string, negative: string, __index_level_0__: int64>)

In [21]:
train_df.head()

,anchor,positive,negative
0,what safeguards are in place to protect the in...,information we collect from other sources we m...,c. the obligations specified in this article s...
1,is there a guarantee from the manufacturers re...,each of the suppliers warrants that the produc...,3.2 manufacturing standards the manufacturer c...
2,what type of authorization has the video confe...,skype hereby grants to online bvi and the comp...,planetcad hereby grants to dassault systemes a...
3,can the blockchain administrator arrange for t...,(a) the fund hereby employs the blockchain adm...,1. duties of the blockchain administrator
4,what happens if a party fails to retain record...,each party will retain such records for at lea...,each of astellas and fg shall retain its recor...


In [22]:
from datasets import load_dataset, concatenate_datasets
test_dataset = load_dataset("json", data_files="../data/tst.json", split="train")
test_dataset=test_dataset.select_columns(["anchor","positive"])
new_column_vals = [""] * len(test_dataset)
test_dataset=test_dataset.add_column("negative",new_column_vals)
# train_dataset = load_dataset("json", data_files="../data/trn_with_hard_negatives.json")
from datasets import Dataset
train_df=pd.read_json('../data/trn_with_hard_negatives.json')
train_dataset = Dataset.from_pandas(train_df)

# assert train_dataset.features.type == test_dataset.features.type
corpus_dataset = concatenate_datasets([train_dataset, test_dataset])

In [24]:
len(corpus_dataset)

37899

### The above column will blow up the size of trn_with_hard_negatives.json, use an indexing scheme instead?

In [154]:
indexer=index_values('../data/indexer.json')
indexer.set(positives)
# indexer.getval(0)
# indexer.getindex('2. the contractor shall not, without the state’s prior written consent, copy, disclose, publish, release, transfer, disseminate, use, or allow access for any purpose or in any form, any confidential information except for the sole and exclusive purpose of performing under the contract.')


# unique_pos=list(set(positives))
# indexer={positive:index for index,positive in enumerate(unique_pos)}
# indexer_reverse={index:positive for positive,index in indexer.items()}

In [156]:
def indx_row(row,col,indxr):
    l=[]
    for positive in row[col]:
        l.append( indxr[positive])
    return l

# df1=df.head()
df['hardnegatives_indexed']=df.apply(indx_row,args=('hardnegatives',indexer.indexer),axis=1)
df['hardnegatives_reverse_indexed']=df.apply(indx_row,args=('hardnegatives_indexed',indexer.indexer_reverse),axis=1)

# df1['hardnegatives_reversed']=df1.apply(indx_row,axis=1)

In [157]:
len(df)
df.tail()

,anchor,positive,hardnegatives,hardnegatives_indexed,hardnegatives_reverse_indexed
35253,for what duration must the verticalnet branded...,2.1. hosting and maintenance. leadersonline sh...,[5.1. access license. subject to the limitatio...,"[2516, 2730, 1295, 2735, 1531, 1308, 1718, 319...",[5.1. access license. subject to the limitatio...
35254,who controls third-party cookies?,it is understood that this policy covers the u...,[tracking technologies we and our marketing an...,"[4, 18, 1698, 32, 2550, 47, 863, 866, 72, 1751...",[tracking technologies we and our marketing an...
35255,detail 'information' storage measures.,ii. information we collect in providing our se...,[the definition of information shall not inclu...,"[2939, 859, 1344, 1845, 1007, 2299, 3165, 2723...",[the definition of information shall not inclu...
35256,what does google expressly warrant regarding t...,8.2 google warrants that the distribution prod...,[2.6 updated versions of distribution products...,"[1017, 1694, 1296, 1537, 269, 2999, 3262, 2198...",[2.6 updated versions of distribution products...
35257,what are the consequences of not updating or p...,a11: bandai namco may amend this privacy polic...,[vendor will maintain and retain the records s...,[8],[vendor will maintain and retain the records s...


In [145]:
# df=df[df['hardnegatives'].apply(lambda x: len(x) == 0)]
# df.drop(columns=['hardnegatives','hardnegatives_indexed','hardnegatives_reverse_indexed'],inplace=True)
# df

In [146]:
# df[df['hardnegatives'].apply(lambda x: len(x) == 0)]

In [77]:
scores[15,:]

tensor([0.4512, 0.1724, 0.2356,  ..., 0.2604, 0.1853, 0.1980], device='cuda:0')

In [27]:
print(df1.iloc[0,2])
print(df1.iloc[0,4])

['if a party or any third party to whom such party has provided confidential information becomes legally compelled (by oral question, deposition, interrogatory, request for documents, subpoena, civil investigative demand or similar process or by rule, regulation or other applicable law) to disclose any confidential information, such party shall promptly notify the other party of such requirement before any disclosure is made so that the other party may seek a protective order or other appropriate remedy or may waive compliance with the terms of this agreement.', '2.5 neither party shall be required to keep confidential any information which is, or becomes, publicly available, is independently developed by either party outside the scope of this agreement, or is rightfully obtained from third parties.', '3. confidentiality the parties hereto agree that each shall treat confidentially the terms and conditions of this agreement and all information provided by each party to the other regard

In [ ]:
# %%time
# # Find the closest 5 sentences of the corpus for each query sentence based on cosine similarity
# top_k = min(100, len(positive_embeddings))

# #for just 1 query
# anchor=anchor_embeddings[0]
# # We use cosine-similarity and torch.topk to find the highest 5 scores
# similarity_scores = model.similarity(anchor, positive_embeddings)[0]
# scores, indices = torch.topk(similarity_scores, k=top_k)

# gt90=0
# gt80=0
# gt70=0
# lt70=0
# for anchor in anchor_embeddings:
#     # We use cosine-similarity and torch.topk to find the highest 5 scores
#     similarity_scores = model.similarity(anchor, positive_embeddings)[0]
#     scores, indices = torch.topk(similarity_scores, k=top_k)
#     if scores[0]>90: 
#         gt90+=1 
#     elif scores[0]>80: 
#         gt80+=1 
#     elif scores[0]>70: 
#         gt70+=1 
#     else: 
#         lt70+=1
# print(f"gt90: {gt90}, gt80: {gt80}, gt70: {gt70}, lt70: {lt70}")
    


gt90: 0, gt80: 0, gt70: 0, lt70: 35258
CPU times: user 14.1 s, sys: 10.3 ms, total: 14.1 s
Wall time: 14.2 s
